<a href="https://colab.research.google.com/github/Elvis-Kayonga/Data-Preprocessing-Formative2-ML-GROUP9/blob/main/product_recommendation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix
import joblib
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

In [55]:
import pandas as pd

# Load datasets
social_data = pd.read_csv("/content/drive/MyDrive/datasets/customer_social_profiles.csv")
trans_data = pd.read_csv("/content/drive/MyDrive/datasets/customer_transactions.csv")

print("Social Data:")
print(social_data.head())

print("\nTransaction Data:")
print(trans_data.head())

Social Data:
  customer_id_new social_media_platform  engagement_score  \
0            A178              LinkedIn                74   
1            A190               Twitter                82   
2            A150              Facebook                96   
3            A162               Twitter                89   
4            A197               Twitter                92   

   purchase_interest_score review_sentiment  
0                      4.9         Positive  
1                      4.8          Neutral  
2                      1.6         Positive  
3                      2.6         Positive  
4                      2.3          Neutral  

Transaction Data:
   customer_id_legacy  transaction_id  purchase_amount purchase_date  \
0                 151            1001              408    2024-01-01   
1                 192            1002              332    2024-01-02   
2                 114            1003              442    2024-01-03   
3                 171            1004

In [56]:
# Convert IDs to same format

social_data['customer_id'] = social_data['customer_id_new'].str.replace('A', '').astype(int)
trans_data['customer_id'] = trans_data['customer_id_legacy']

# Merge datasets
merged_data = pd.merge(social_data, trans_data, on='customer_id', how='inner')

print("\nMerged Data:")
print(merged_data.head())

print("\nInfo:")
merged_data.info()

print("\nShape:", merged_data.shape)


Merged Data:
  customer_id_new social_media_platform  engagement_score  \
0            A190               Twitter                82   
1            A190               Twitter                82   
2            A150              Facebook                96   
3            A150              Facebook                96   
4            A162               Twitter                89   

   purchase_interest_score review_sentiment  customer_id  customer_id_legacy  \
0                      4.8          Neutral          190                 190   
1                      4.8          Neutral          190                 190   
2                      1.6         Positive          150                 150   
3                      1.6         Positive          150                 150   
4                      2.6         Positive          162                 162   

   transaction_id  purchase_amount purchase_date product_category  \
0            1031              333    2024-01-31        Groceries   


In [ ]:
merged_data.to_csv("/content/drive/MyDrive/datasets/merged_dataset.csv", index=False)
print("Merged dataset saved!")

Merged dataset saved!


#Data Cleaning and processing

In [57]:
# Fill missing ratings with median
merged_data['customer_rating'].fillna(merged_data['customer_rating'].median(), inplace=True)

# Check nulls
merged_data.isnull().sum()

/tmp/ipykernel_1081/2435663639.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_data['customer_rating'].fillna(merged_data['customer_rating'].median(), inplace=True)


,0
customer_id_new,0
social_media_platform,0
engagement_score,0
purchase_interest_score,0
review_sentiment,0
customer_id,0
customer_id_legacy,0
transaction_id,0
purchase_amount,0
purchase_date,0


In [58]:

#  Transform date
merged_data['purchase_date'] = pd.to_datetime(merged_data['purchase_date'])

merged_data['purchase_month'] = merged_data['purchase_date'].dt.month
merged_data['purchase_day'] = merged_data['purchase_date'].dt.day

# Drop useless columns
merged_data = merged_data.drop([
    'customer_id_new',
    'customer_id',
    'customer_id_legacy',
    'transaction_id',
    'purchase_date'
], axis=1)

#  Separate target
y = merged_data['product_category']
X = merged_data.drop('product_category', axis=1)

# 5. Encode categorical columns
X = pd.get_dummies(X, columns=[
    'social_media_platform',
    'review_sentiment'
]).astype(int)

print(X.head())

joblib.dump(X.columns, "/content/drive/MyDrive/datasets/model_columns.pkl")
print("model_columns.pkl saved successfully")

   engagement_score  purchase_interest_score  purchase_amount  \
0                82                        4              333   
1                82                        4              401   
2                96                        1              389   
3                96                        1              177   
4                89                        2              101   

   customer_rating  purchase_month  purchase_day  \
0                3               1            31   
1                4               5            19   
2                3               2            11   
3                3               2            15   
4                4               3            19   

   social_media_platform_Facebook  social_media_platform_Instagram  \
0                               0                                0   
1                               0                                0   
2                               1                                0   
3               

In [59]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Encode target variable
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Save the label encoder
joblib.dump(label_encoder, "/content/drive/MyDrive/datasets/product_label_encoder.pkl")
print("product_label_encoder.pkl saved successfully")


product_label_encoder.pkl saved successfully


#Train XGB Model

In [68]:
# Define XGBoost classifier
model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    max_depth=5,
    learning_rate=0.1,
    n_estimators=200,
    random_state=42,
    eval_metric='mlogloss'
)

# Train model
model.fit(X_train, y_train_encoded)

# Predict
y_pred_encoded = model.predict(X_test)
y_pred = label_encoder.inverse_transform(y_pred_encoded)

# Save the model
joblib.dump(model, "/content/drive/MyDrive/datasets/product_xgb_model.pkl")
print("product_xgb_model.pkl saved successfully")

product_xgb_model.pkl saved successfully


In [69]:
# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.6590909090909091
F1 Score: 0.6569937239793698

Classification Report:
               precision    recall  f1-score   support

       Books       0.56      0.50      0.53        10
    Clothing       0.67      0.50      0.57         4
 Electronics       0.58      0.78      0.67         9
   Groceries       0.70      0.78      0.74         9
      Sports       0.80      0.67      0.73        12

    accuracy                           0.66        44
   macro avg       0.66      0.64      0.65        44
weighted avg       0.67      0.66      0.66        44

[[5 0 3 2 0]
 [0 2 0 0 2]
 [0 1 7 1 0]
 [2 0 0 7 0]
 [2 0 2 0 8]]
